# CLM-0.3 — Progressive Growth

Implementation/preflight notebook. The formal GPU experiment is intentionally disabled until the preflight gates are reviewed.

In [ ]:
from pathlib import Path
import json, sys, hashlib
ROOT = Path('/kaggle/working/mini-cells')
sys.path.insert(0, str(ROOT / 'research'))
print('research path:', sys.path[0])

In [ ]:
import torch
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
release = ROOT / 'artifacts/releases/clm-0.1/model.pt'
observed = hashlib.sha256(release.read_bytes()).hexdigest()
assert observed == '87d36c408ae3873ffd567ebf17050661b42ddae2c8d5d1bab84b2c27c3c7e7a0'
print('CLM-0.1 SHA-256:', observed)

In [ ]:
# CPU/preflight checks only; do not run the formal matrix in this cell.
import subprocess
subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_clm_progressive_growth.py', 'tests/test_growth_router.py', 'tests/test_growth_checkpoint.py', '-q'], cwd=ROOT, check=True)

In [ ]:
from minicells.clm_growth import ProgressiveGrowthCLM
model = ProgressiveGrowthCLM.from_clm01_release(str(ROOT / 'artifacts/releases/clm-0.1'))
print('zero-birth expert counts:', model.expert_counts_by_stage())
print('formal GPU experiment: disabled')

The remaining experiment cells should be enabled only after reviewing the baseline, parity, checkpoint/resume, pressure-table, and telemetry preflight outputs. Publishing remains disabled by default.

In [ ]:
# Explicit safety switch: changing this to True starts all nine formal GPU workers.
RUN_FORMAL = False
RESULTS = ROOT / 'results/clm-0.3-progressive-growth'
runner = [sys.executable, 'scripts/run_clm_progressive_growth_001.py', '--output-root', str(RESULTS)]
if RUN_FORMAL:
    subprocess.run([*runner, '--execute'], cwd=ROOT, check=True)
else:
    subprocess.run(runner, cwd=ROOT, check=True)
    print('NO DATA / PREFLIGHT ONLY')

In [ ]:
# Aggregate only completed worker artifacts; never synthesize missing curves.
import csv
rows = []
for path in sorted(RESULTS.glob('r*-*/ppl-history.csv')):
    with path.open() as handle:
        rows.extend(csv.DictReader(handle))
if rows:
    print('formal PPL rows:', len(rows))
else:
    print('NO DATA / PREFLIGHT ONLY; plots and formal decisions skipped')